For checking pipeline against selected low-pass ARTEMIS data.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import numpy as np
import pandas as pd
from lunarmag import run_passes

REPO = Path.cwd().resolve().parent
RAW, DERIVED = REPO / 'data' / 'raw', REPO / 'data' / 'derived'
DERIVED.mkdir(parents=True, exist_ok=True)

/Users/yulun/artemis-lunar-plasma/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load pass list from raw data file:

In [2]:
passes = pd.read_csv(RAW / 'thb_2012_2025_all_passes.csv')
passes['pass_id'] = passes['year'].astype(str) + '-' + passes['pass_number'].astype(str).str.zfill(2)
passes['t_peri'] = pd.to_datetime(passes['lowest_minute_utc']).dt.strftime('%Y-%m-%d/%H:%M:%S')

print(len(passes), 'passes; IDs unique:', passes['pass_id'].is_unique)
passes[['pass_id', 't_peri', 'lowest_altitude_km', 'latitude_deg', 'longitude_deg', 'sza_deg', 'phi_gse_deg']].head()

142 passes; IDs unique: True


,pass_id,t_peri,lowest_altitude_km,latitude_deg,longitude_deg,sza_deg,phi_gse_deg
0,2012-01,2012-01-19/20:25:00,38.5,9.05,274.32,47.76,-44.51
1,2012-02,2012-01-20/23:34:00,79.3,8.81,261.57,48.74,-29.79
2,2012-03,2012-02-02/13:30:00,67.0,10.09,89.02,30.14,116.05
3,2012-04,2012-02-15/03:18:00,49.2,10.49,280.95,16.26,-84.46
4,2012-05,2012-02-16/06:28:00,46.7,10.70,263.66,13.63,-69.89


List for overview of data:

In [3]:
print(passes.groupby('year').size().to_dict())
passes[['lowest_altitude_km', 'sza_deg', 'latitude_deg', 'duration_min']].describe().round(1)

{2012: 18, 2013: 15, 2014: 11, 2015: 8, 2016: 11, 2017: 9, 2018: 7, 2019: 9, 2020: 10, 2021: 6, 2022: 13, 2023: 9, 2024: 6, 2025: 10}


,lowest_altitude_km,sza_deg,latitude_deg,duration_min
count,142.0,142.0,142.0,142.0
mean,77.6,59.1,0.5,4.8
std,18.6,26.8,7.5,2.5
min,14.8,8.5,-10.9,1.0
25%,69.2,38.0,-7.1,3.0
50%,82.0,65.8,1.3,5.0
75%,90.7,84.4,7.9,6.0
max,99.7,89.9,10.9,12.0


Trial run on three passes first:

In [4]:
trial_ids = ['2013-12', '2013-13', '2013-14']
trial = passes[passes['pass_id'].isin(trial_ids)]

results, failures = run_passes(trial['t_peri'].tolist(), ids=trial['pass_id'])
failures

23-Nov-26 13:07:07: Downloading https://themis.ssl.berkeley.edu/data/themis/thb/l2/fgm/2013/thb_l2_fgm_20131202_v01.cdf to themis_data/thb/l2/fgm/2013/thb_l2_fgm_20131202_v01.cdf
23-Nov-26 13:07:09: Download of themis_data/thb/l2/fgm/2013/thb_l2_fgm_20131202_v01.cdf complete, 14.997 MB in 2.5 sec (6.102 MB/sec) (transfer_normal)
23-Nov-26 13:07:09: Downloading https://themis.ssl.berkeley.edu/data/themis/thc/l2/fgm/2013/thc_l2_fgm_20131202_v01.cdf to themis_data/thc/l2/fgm/2013/thc_l2_fgm_20131202_v01.cdf
23-Nov-26 13:07:11: Download of themis_data/thc/l2/fgm/2013/thc_l2_fgm_20131202_v01.cdf complete, 18.802 MB in 2.2 sec (8.694 MB/sec) (transfer_normal)
23-Nov-26 13:07:12: time_clip: thc_fgh_btotal has no data in requested range
23-Nov-26 13:07:12: time_clip: thc_fgh_gse has no data in requested range
23-Nov-26 13:07:12: time_clip: thc_fgh_gsm has no data in requested range
23-Nov-26 13:07:12: time_clip: thc_fgh_dsl has no data in requested range
23-Nov-26 13:07:12: time_clip: thc_fgh_

1/3 done, 0 failed

23-Nov-26 13:07:26: File is current: themis_data/thb/l2/fgm/2013/thb_l2_fgm_20131203_v01.cdf
23-Nov-26 13:07:27: File is current: themis_data/thb/l2/fgm/2013/thb_l2_fgm_20131204_v01.cdf
23-Nov-26 13:07:27: File is current: themis_data/thc/l2/fgm/2013/thc_l2_fgm_20131203_v01.cdf
23-Nov-26 13:07:28: File is current: themis_data/thc/l2/fgm/2013/thc_l2_fgm_20131204_v01.cdf
23-Nov-26 13:07:29: time_clip: thc_fgh_btotal has no data in requested range
23-Nov-26 13:07:29: time_clip: thc_fgh_gse has no data in requested range
23-Nov-26 13:07:29: time_clip: thc_fgh_gsm has no data in requested range
23-Nov-26 13:07:30: time_clip: thc_fgh_dsl has no data in requested range
23-Nov-26 13:07:30: time_clip: thc_fgh_ssl has no data in requested range
23-Nov-26 13:07:30: File is current: themis_data/thb/l1/state/2013/thb_l1_state_20131203.cdf
23-Nov-26 13:07:31: File is current: themis_data/thb/l1/state/2013/thb_l1_state_20131204.cdf
23-Nov-26 13:07:31: Creating spin model for probe b correction level 

2/3 done, 0 failed

23-Nov-26 13:07:48: File is current: themis_data/thb/l2/fgm/2013/thb_l2_fgm_20131205_v01.cdf
23-Nov-26 13:07:49: File is current: themis_data/thc/l2/fgm/2013/thc_l2_fgm_20131205_v01.cdf
23-Nov-26 13:07:50: time_clip: thc_fgh_btotal has no data in requested range
23-Nov-26 13:07:50: time_clip: thc_fgh_gse has no data in requested range
23-Nov-26 13:07:50: time_clip: thc_fgh_gsm has no data in requested range
23-Nov-26 13:07:50: time_clip: thc_fgh_dsl has no data in requested range
23-Nov-26 13:07:50: time_clip: thc_fgh_ssl has no data in requested range
23-Nov-26 13:07:50: Eclipse correction status for probe ['b', 'c']:
23-Nov-26 13:07:50: Eclipse 1 of 3: start: 2013-12-03 00:00:01.401909  end: 2013-12-03 00:23:43.143876 status: in shadow, partially corrected (field waveforms corrected; particle data and EFI or FGM spin fits will have uncorrected spin phase offsets)
23-Nov-26 13:07:50: Eclipse 2 of 3: start: 2013-12-04 00:20:12.474808  end: 2013-12-04 01:21:21.007147 status: in shadow, 

3/3 done, 0 failed

""


Success! Check against data columns:

In [5]:
table = results.merge(passes, on='pass_id', suffixes=('', '_margot'))
table[['pass_id', 'min_alt_km', 'lowest_altitude_km', 'tail_angle_deg', 'phi_gse_deg', 'regime_geo', 'bg_ok', 'bg_sigma', 'post_status', 'n_used', 'peak_dB', 'peak_dBi']]

,pass_id,min_alt_km,lowest_altitude_km,tail_angle_deg,phi_gse_deg,regime_geo,bg_ok,bg_sigma,post_status,n_used,peak_dB,peak_dBi
0,2013-12,33.151552,33.5,175.613516,-0.66,solar_wind,False,0.911649,disturbed,248,2.988227,2.701883
1,2013-13,14.786443,14.8,161.731000,13.85,solar_wind,True,0.399926,disturbed,248,6.515042,5.717913
2,2013-14,86.387652,86.7,147.219551,28.47,solar_wind,False,0.149600,disturbed,276,3.356956,3.281770


In [6]:
stamp = pd.Timestamp.today().strftime('%Y-%m-%d')
results.to_csv(DERIVED / f'trial_results_{stamp}.csv', index=False)
failures.to_csv(DERIVED / f'trial_failures_{stamp}.csv', index=False)

All three passes asssociated with disturbed post_status and two had failed backgrounds. Check how tests failed:

In [7]:
table[['pass_id', 'bg_sigma', 'bg_r', 'bg_shift', 'post_window', 'post_sigma', 'post_shift', 'shadow_s']]

,pass_id,bg_sigma,bg_r,bg_shift,post_window,post_sigma,post_shift,shadow_s
0,2013-12,0.911649,0.754562,120.0,"(1386026190.0, 1386026370.0)",0.474022,-38.0,780.0
1,2013-13,0.399926,0.940293,12.0,"(1386116100.0, 1386116280.0)",2.050849,16.0,780.0
2,2013-14,0.149600,0.735600,-23.0,"(1386206040.0, 1386206220.0)",0.897016,45.0,600.0


2013-12: fails on noise and delay (bg_sigma and ng_shift, 120 at edge); also shift is negative, i.e. THC downstream from THB.

2013-13: passes!

2013-14: fails since we set r_min = 0.8 in background.py. Changed to 0.5 instead.